# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/10Plaiz/flyrank-ml-i-starter/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
# Setup — works in Colab and locally.
%pip install -q pandas
import os, subprocess

REPO = "flyrank-ml-i-starter"          # personal copy. The Colab badges point at the same one
if not os.path.exists("data/raw/content_refresh_anonymized.csv"):
    if not os.path.isdir(REPO):        # running in Colab: bring the repo in
        subprocess.run(["git", "clone", "--depth", "1",
                        f"https://github.com/10Plaiz/{REPO}.git"], check=True)
    os.chdir(REPO)

import pandas as pd
pd.set_option("display.width", 110)
# Deliberately not printing the full working directory — this notebook is public.
print("starter data found:", os.path.exists("data/raw/content_refresh_anonymized.csv"))
print("pandas", pd.__version__)



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
starter data found: True
pandas 2.3.3


## 1. My lane (or freestyle) and why

**AI Referral Opportunity direction**

FlyRank's core problem is deciding which page a human should fix first. Search referrals are the
well-lit part of that problem; AI-assistant referrals are the part nobody has a rule for yet and
the lane guide is explicit that this direction is sparse and easy to overclaim. That combination is
why I want it: the interesting work is not fitting something, it is establishing what this column
can honestly support before anyone builds a queue on top of it.

The sparsity warning in §9 is stated at daily grain (30,177 AI rows against 78.8M). The starter
slice aggregates to 90 days per page, and at that grain the coverage is workable as the cell below
is my check that the direction is viable before I commit a week to it.

In [2]:
# Is the AI-referral direction viable on the starter slice at all?
#
# Finding: 1,930 of 30,000 pages (6.43%) carry at least one AI referral, spread across
# 26 of the 32 clients. Thin, but workable — and the reason it is workable is grain.
# The lane guide's sparsity warning is stated at DAILY grain (30,177 AI rows against
# 78.8M); aggregating to 90 days per page is what lifts coverage into usable range.
#
# Finding: the magnitudes are tiny — median 2 AI sessions per page, 90th percentile 7.
# That is why every rate later in this notebook needs a volume floor before it means
# anything at all.
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["has_ai"] = (df["ai_sessions_90d"].fillna(0) > 0).astype(int)

n_pages, n_ai = len(df), int(df["has_ai"].sum())
print(f"pages in the starter slice : {n_pages:,}")
print(f"pages with any AI referral : {n_ai:,}  ({n_ai / n_pages:.2%})")
print(f"clients represented        : {df['client_id'].nunique()}"
      f"   (with >=1 AI page: {df.loc[df['has_ai'] == 1, 'client_id'].nunique()})")

print("\nAI sessions per page, among the pages that have any:")
print(df.loc[df["has_ai"] == 1, "ai_sessions_90d"]
        .describe(percentiles=[.5, .75, .9]).round(2).to_string())


pages in the starter slice : 30,000
pages with any AI referral : 1,930  (6.43%)
clients represented        : 32   (with >=1 AI page: 26)

AI sessions per page, among the pages that have any:
count    1930.00
mean        3.18
std         4.41
min         1.00
50%         2.00
75%         3.00
90%         7.00
max        64.00


## 2. The question: decision, action, cost of a wrong call

> **For a content editor with capacity to review about 20 pages a week, which pages capture less
> AI-assistant referral traffic than comparable pages at the same exposure level — and is that gap
> a real opportunity or a measurement artifact?**

- **Decision improved:** which pages to put in front of an editor first for AI-visibility work.
- **Who acts, and how:** a content editor works a weekly ranked queue, top down.
- **Cost of a wrong call:** wasted editor hours on a page whose low AI share is really just low
  exposure, a tiny denominator, or noise. The cost is asymmetric — a missed opportunity is
  invisible, but a queue full of false positives is how an editor stops trusting the queue.
- **Why a plain rule is not enough:** ranking by raw AI share ranks by traffic volume, which the
  editor already knows. The signal only appears once pages are compared within an exposure
  stratum — and that comparison is the thing I have to build and defend.

In [3]:
# The core evidence behind my question: AI referral tracks EXPOSURE, not content character.
# Volume floor first — a share computed on a handful of sessions is noise.
#
# Finding: the share of pages with any AI referral falls with every step down the
# exposure ladder, ~30% at "excellent" to ~3% at "low" — roughly a 9x spread.
#
# Consequence, and the reason my question is worded the way it is: ranking pages by raw
# AI share would mostly re-rank them by traffic, which the editor already knows. Pages
# have to be compared like with like — each against others in its own exposure tier.
measurable = df[df["sessions_90d"] >= 10]

gradient = (measurable.groupby("impression_tier")
            .agg(pages=("has_ai", "size"),
                 pages_with_ai=("has_ai", "sum"),
                 pct_with_ai=("has_ai", "mean")))
gradient["pct_with_ai"] = (gradient["pct_with_ai"] * 100).round(1)
gradient = gradient.reindex(["excellent", "good", "moderate", "low"])

print("pages with any AI referral, by exposure tier (sessions_90d >= 10):\n")
print(gradient.to_string())

ratio = gradient.loc["excellent", "pct_with_ai"] / gradient.loc["low", "pct_with_ai"]
print(f"\ntop-tier / bottom-tier ratio: {ratio:.1f}x")


pages with any AI referral, by exposure tier (sessions_90d >= 10):

                 pages  pages_with_ai  pct_with_ai
impression_tier                                   
excellent         1071            327         30.5
good              6490           1014         15.6
moderate          4693            252          5.4
low               1368             46          3.4

top-tier / bottom-tier ratio: 9.0x


## 3. Quick look at the data (2-3 real numbers)

Three numbers decide the shape of this project. The exposure gradient is in section 2 above; these
are the three that constrain *how* I am allowed to use it.

1. **Concentration** — if one client dominates the AI pages, a portfolio-level finding is really a
   single-site finding. This sets whether I stratify by client or rank within it.
2. **The denominator artifact** — `ai_traffic_pct` can exceed 100 by design, and I need to know
   whether that is measurement noise (fixable with a floor) or a data error (not fixable).
3. **The uncontrolled comparison** — what I would have concluded if I had skipped the exposure
   control, so the write-up can show the trap rather than just claim to have avoided it.

In [4]:
# [1] Concentration — is this a portfolio finding, or one client's story?
#
# Finding: the largest single client holds 938 of the 1,930 AI pages (48.6%), so any
# portfolio-level claim risks being one site's behaviour wearing a portfolio costume.
# 20 clients still clear a >=30 pages / >=10 AI pages bar, with AI rates spanning
# 1.6%-25.5% (median 3.7%) — enough spread to stratify by client rather than pool.
per_client = (df.groupby("client_id")
                .agg(pages=("has_ai", "size"), ai_pages=("has_ai", "sum")))
per_client["ai_rate_pct"] = (per_client["ai_pages"] / per_client["pages"] * 100).round(1)
usable = per_client[(per_client["pages"] >= 30) & (per_client["ai_pages"] >= 10)]

biggest, total_ai = per_client["ai_pages"].max(), per_client["ai_pages"].sum()
print(f"[1] largest client holds {biggest:,} of {total_ai:,} AI pages "
      f"({biggest / total_ai:.1%})")
print(f"    clients with >=30 pages AND >=10 AI pages: {len(usable)}")
print(f"    their AI rate spans {usable['ai_rate_pct'].min()}% - {usable['ai_rate_pct'].max()}%"
      f"  (median {usable['ai_rate_pct'].median()}%)")

# [2] The denominator artifact that sets my volume floor.
#
# ai_traffic_pct = ai_sessions_90d / sessions_90d x 100, and the numerator and denominator
# are measured by different systems, so a tiny denominator produces an impossible-looking
# rate. Finding: 23 rows exceed 100%, and 19 of those have fewer than 10 sessions. That is
# a floor to set, not a bug to fix — a sessions_90d >= 30 floor keeps 1,357 AI pages.
over = df[df["ai_traffic_pct"] > 100]
floor_pop = int(((df["has_ai"] == 1) & (df["sessions_90d"] >= 30)).sum())
print(f"\n[2] rows with ai_traffic_pct > 100: {len(over)}"
      f"   (of which sessions_90d < 10: {int((over['sessions_90d'] < 10).sum())})")
print(f"    AI pages surviving a sessions_90d >= 30 floor: {floor_pop:,}")

# [3] What I would have concluded if I had skipped the exposure control.
#
# Finding: AI pages carry ~12.7x the impressions and ~12.1x the sessions of non-AI pages,
# while age (229 vs 236 days) and informational share (61.2% vs 62.4%) barely move. So the
# obvious story — "AI prefers long informational pages" — is mostly exposure in disguise.
# Keeping this comparison in the notebook lets the write-up SHOW the trap rather than
# just claim to have avoided it.
cols = ["impressions_90d", "sessions_90d", "word_count", "content_age_days"]
med = df.groupby("has_ai")[cols].median().T
med.columns = ["no_ai", "has_ai"]
med["ratio"] = (med["has_ai"] / med["no_ai"]).round(2)
print("\n[3] median page, AI vs non-AI, with NO control for exposure:\n")
print(med.to_string())

ai_int = df.loc[df["has_ai"] == 1, "main_intent"].value_counts(normalize=True)
all_int = df["main_intent"].value_counts(normalize=True)
print(f"\n    informational share - AI pages {ai_int.get('informational', 0):.1%}"
      f"  vs all pages {all_int.get('informational', 0):.1%}")


[1] largest client holds 938 of 1,930 AI pages (48.6%)
    clients with >=30 pages AND >=10 AI pages: 20
    their AI rate spans 1.6% - 25.5%  (median 3.7%)

[2] rows with ai_traffic_pct > 100: 23   (of which sessions_90d < 10: 19)
    AI pages surviving a sessions_90d >= 30 floor: 1,357

[3] median page, AI vs non-AI, with NO control for exposure:

                   no_ai  has_ai  ratio
impressions_90d    630.5  8014.5  12.71
sessions_90d         7.0    85.0  12.14
word_count        2844.0  4395.0   1.55
content_age_days   236.0   229.0   0.97

    informational share - AI pages 61.2%  vs all pages 62.4%


## 4. Careful words: what I can and can't claim

This column measures **click-throughs from AI assistants**, nothing more. Every limit below follows
from that one fact.

**What this work will be able to say.** That I *observed* differences in AI referral share between
pages at comparable exposure; that the gap is *associated with* measurable page attributes; that a
ranked queue puts plausible review candidates in front of an editor first. Decision-support, stated
as observed and directional.

**What it will never say.** That anything *caused* AI referrals — I have no experiment and no causal
design. That a page with no AI sessions is invisible to or misunderstood by AI systems — absence of
a click-through is not absence of a citation, and the data cannot distinguish them. Anything about
Google's algorithm, AI rankings, or AI citations, all of which §10 rules out.

**The honest boundary of the population.** Everything I claim applies to pages that clear the volume
floor below, inside this 90-day anonymized slice — not to the web, not to AI systems in general.

In [5]:
# The population this project actually speaks about, after the honest floors.
#
# What ai_sessions_90d counts (docs/data-dictionary.md): GA4 sessions referred from AI
# tools — click-throughs from AI assistants. NOT citations, NOT rankings, NOT "AI
# visibility". Every claim limit written in section 4 above follows from that one fact.
#
# Finding: inside the sessions_90d >= 30 floor the AI referral rate is 19.1%, about three
# times the unfiltered 6.43%. The floor is removing pages that were never measurable,
# not pages that failed — which is exactly why the floor belongs in the claim, not just
# in the code. This is the population every statement in this project is scoped to.
population = df[df["sessions_90d"] >= 30]
print(f"analysis population (sessions_90d >= 30) : {len(population):,} pages")
print(f"  showing any AI referral                : {int(population['has_ai'].sum()):,}"
      f"  ({population['has_ai'].mean():.1%})")
print(f"  clients represented                    : {population['client_id'].nunique()}")


analysis population (sessions_90d >= 30) : 7,114 pages
  showing any AI referral                : 1,357  (19.1%)
  clients represented                    : 28


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.